# Análisis Exploratorio de Datos con el dataset de diamantes

En este notebook aprenderás a realizar un análisis exploratorio de datos (EDA) utilizando una base de datos SQLite con información sobre diamantes. Exploraremos tipos de variables, escalas de medición, estadísticas descriptivas, visualizaciones y transformaciones útiles para preparar los datos para modelos de machine learning.

---

### Conexión a la base de datos

### ¿Qué es Git?

Git es una herramienta que permite gestionar versiones de archivos, especialmente útil en proyectos de programación. Con Git puedes:

- Guardar cambios progresivos en tu trabajo
- Colaborar con otras personas sin perder el control de versiones
- Descargar (clonar) proyectos públicos desde plataformas como GitHub

En este notebook usamos `git clone` para copiar un repositorio que contiene bases de datos educativas en formato SQLite.

### ¿Qué es SQL?

SQL (Structured Query Language) es un lenguaje utilizado para consultar y manipular bases de datos. Permite extraer, filtrar, combinar y ordenar información de forma estructurada.

En este notebook usamos SQL para obtener datos de una base de datos SQLite. Algunas cláusulas clave que utilizamos son:

- `SELECT`: indica qué columnas queremos ver
- `FROM`: especifica de qué tabla se obtienen los datos
- `JOIN`: combina información de varias tablas relacionadas
- `ON`: define la condición de unión entre tablas
- `LIMIT`: restringe la cantidad de filas que se muestran (opcional)

Ejemplo usado:
```sql
SELECT carat, price, cut
FROM Observation
JOIN Cut ON Observation.cut_id = Cut.cut_id

# Generar primer SQL query.

In [ ]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/diamonds.db"
r = requests.get(url)

with open("diamonds.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("diamonds.db")

query = """
SELECT
  O.carat,
  O.price,
  O.depth,
  "O"."table",
  O.x,
  O.y,
  O.z,
  C.cut,
  Co.color,
  Cl.clarity
FROM
  Observation AS O
JOIN
  Cut AS C ON O.cut_id = C.cut_id
JOIN
  Color AS Co ON O.color_id = Co.color_id
JOIN
  Clarity AS Cl ON O.clarity_id = Cl.clarity_id
"""

df = pd.read_sql_query(query, conn)
df.head()

## Descripción de las columnas del dataset `diamonds`

La calidad de un diamante se evalúa principalmente mediante las **4 C's** (por sus siglas en inglés): Carat (quilate), Cut (corte), Clarity (claridad) y Color. A continuación se describen todas las columnas del dataset.

---

### 💎 Carat (Quilate)
- **Tipo:** Numérico (float)
- **Escala:** Razón
- **Descripción:** Peso del diamante. Un quilate equivale a 200 mg. A mayor peso, generalmente mayor precio. Si dos diamantes tienen el mismo peso, se usan las otras características para determinar el precio.

---

### ✂️ Cut (Corte)
- **Tipo:** Categórico (ordinal)
- **Escala:** Ordinal
- **Descripción:** Calidad del corte del diamante. Mide tres aspectos fundamentales:
  - **Brillo (Brilliance):** Reflejo de luz blanca dentro y fuera del diamante.
  - **Fuego (Fire):** Dispersión de la luz blanca en los colores del arcoíris.
  - **Centelleo (Scintillation):** Cantidad de destellos y patrón de luces/sombras por reflejos internos.

- **Categorías (de menor a mayor calidad):** Fair < Good < Very Good < Premium < Ideal

---

### 🔍 Clarity (Claridad)
- **Tipo:** Categórico (ordinal)
- **Escala:** Ordinal
- **Descripción:** Indica la cantidad de inclusiones (marcas internas) y manchas (blemishes, marcas externas) presentes en el diamante. Los diamantes se forman bajo presión y calor extremos, lo que genera estas imperfecciones.

- **Categorías (de mayor a menor claridad):** FL > IF > VVS1 > VVS2 > VS1 > VS2 > SI1 > SI2 > I1 > I2 > I3

---

### 🎨 Color
- **Tipo:** Categórico (ordinal)
- **Escala:** Ordinal
- **Descripción:** Mide la ausencia de color en el diamante. Un diamante incoloro (como una gota de agua) tiene mayor valor porque dispersa mejor la luz.

- **Escala de color (de mejor a peor):**
  - D, E, F — Incoloro
  - G, H, I, J — Casi incoloro
  - K, L, M — Color tenue
  - N–R — Color muy ligero
  - S–Z — Color ligero (amarillento)

---

### 📐 Depth (Profundidad %)
- **Tipo:** Numérico (float)
- **Escala:** Intervalo (porcentaje)
- **Descripción:** Distancia desde la mesa (superficie superior) hasta la culata (punta inferior), expresada como porcentaje del ancho total.
- **Fórmula:** `depth = z / promedio(x, y) × 100`
- **Nota:** Una profundidad menor hace que el diamante parezca más grande visto desde arriba.

---

### 📏 Table (Mesa %)
- **Tipo:** Numérico (float)
- **Escala:** Intervalo (porcentaje)
- **Descripción:** Ancho de la faceta superior (mesa) como porcentaje del ancho total del diamante. Una mesa adecuada permite que la luz entre y se refleje correctamente.

---

### 📦 Dimensiones: x, y, z
- **Tipo:** Numérico (float)
- **Escala:** Razón (milímetros)
- **Descripción:**
  - `x` — Largo (mm)
  - `y` — Ancho (mm)
  - `z` — Profundidad/alto (mm)
- **Uso:** La relación largo/ancho (`x/y`) determina la forma del diamante:
  - Redondo: ratio entre 1.00 y 1.05
  - Ovalado: ratio ≤ 1.50

---

### 💰 Price (Precio)
- **Tipo:** Numérico (int)
- **Escala:** Razón
- **Descripción:** Precio del diamante en dólares estadounidenses. Es la variable objetivo típica en modelos de predicción.

### Identificar tipos de datos

In [ ]:
df.dtypes

In [ ]:
numericas = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categoricas = df.select_dtypes(include=['object', 'str']).columns.tolist()

print("Variables numéricas:", numericas)
print("Variables categóricas:", categoricas)

### Escala de medición de las variables

| Variable         | Tipo de dato | Escala de medición | Justificación |
|------------------|--------------|---------------------|----------------|
| `carat`          | Numérica     | Razón               | Tiene cero absoluto, se puede multiplicar/dividir |
| `price`          | Numérica     | Razón               | Representa valor monetario, cero tiene significado |
| `x`, `y`, `z`     | Numérica     | Razón               | Medidas físicas, cero indica ausencia |
| `depth`          | Numérica     | Intervalo           | Porcentaje relativo, no tiene cero absoluto claro |
| `"table"`        | Numérica     | Intervalo           | Porcentaje relativo, no tiene cero absoluto claro |
| `cut`            | Categórica   | Ordinal             | Tiene orden lógico: Fair < Good < Very Good < Ideal < Premium |
| `color`          | Categórica   | Ordinal o Nominal           | Escala gemológica: D (mejor) a J (peor) |
| `clarity`        | Categórica   | Ordinal o Nominal           | Escala gemológica: FL > IF > VVS1 > ... > I3 |


### Estadísticas Descriptivas

In [ ]:
df.describe()

In [ ]:
for col in categoricas:
    print(f"\nFrecuencias de '{col}':")
    print(df[col].value_counts())

- ¿Qué tipo de codificación sería más adecuada para cada variable categórica?


### Correlaciones entre variables numéricoas

In [ ]:
import plotly.express as px

corr_matrix = df[numericas].corr().round(2)
fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='Viridis'
)
fig.update_layout(title='Mapa de calor de correlaciones')
fig.show()

- ¿Qué variables parecen tener una relación fuerte con el precio?

### Visualizaciones con plotly

In [ ]:
import plotly.express as px

fig = px.scatter(
    df,
    x='carat',
    y='price',
    color='cut',
    title='Relación entre carat y price según tipo de corte',
    hover_data=['color', 'clarity'],
    trendline='ols',
    trendline_scope='overall'
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='depth',
    y='price',
    color='cut',
    title='Relación entre depth y price según tipo de corte',
    hover_data=['color', 'clarity'],
    trendline='ols',
    trendline_scope='overall'
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='table',
    y='price',
    color='cut',
    title='Relación entre table y price según tipo de corte',
    hover_data=['color', 'clarity'],
    trendline='ols',
    trendline_scope='overall'
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='x',
    y='price',
    color='cut',
    title='Relación entre longitud y price según tipo de corte',
    hover_data=['color', 'clarity'],
    trendline='ols',
    trendline_scope='overall'
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='y',
    y='price',
    color='cut',
    title='Relación entre ancho y price según tipo de corte',
    hover_data=['color', 'clarity'],
    trendline='ols',
    trendline_scope='overall'
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='z',
    y='price',
    color='cut',
    title='Relación entre profunidad y price según tipo de corte',
    hover_data=['color', 'clarity'],
    trendline='ols',
    trendline_scope='overall'
)
fig.show()

**Conclusión del Gráfico:**

Podemos observar que el quilate, la longitud, la anchura y la profundidad muestran una linealidad con el precio, con menos valores atípicos, mientras que el porcentaje de tabla, el porcentaje de profundidad y la relación L/W muestran una linealidad, pero con valores atípicos elevados.

### Boxplot pot tipo de corte

In [ ]:
fig = px.box(
    df,
    x='color',
    y='price',
    color='color',
    category_orders={'color': ['D', 'E', 'F', 'G', 'H', 'I', 'J']},
    title='Distribución de precios por color'
)
fig.show()

**Conclusión del gráfico:**

Del boxplot anterior podemos observar que los colores G, H, I y J tienen menos valores atípicos (outliers) en comparación con los colores D y E. Esto sugiere que, a mayor calidad de color, mayor cantidad de outliers — con excepción del color G. Además, todas las categorías de color presentan precios máximos y mínimos similares.

In [ ]:
fig = px.box(
    df,
    x='cut',
    y='price',
    color='cut',
    category_orders={'cut': ['Ideal' ,'Premium' ,'Very Good' ,'Good' ,'Fair']},
    title='Distribución de precios por tipo de corte'
)
fig.show()

**Conclusión del gráfico:**

Del boxplot anterior podemos observar que, a menor calidad de corte, mayor es la cantidad de valores atípicos (outliers) — con excepción del corte Ideal. Además, todas las categorías de corte presentan precios máximos y mínimos similares.

In [ ]:
fig = px.box(
    df,
    x='clarity',
    y='price',
    color='clarity',
    category_orders={'clarity': ['IF','VVS1','VVS2','VS1','VS2','SI1','SI2','I1']},
    title='Distribución de precios por claridad'
)
fig.show()

**Conclusión del gráfico:**

Del boxplot anterior podemos observar que las categorías IF, VVS1 y VVS2 tienen una alta cantidad de valores atípicos (outliers) en comparación con las demás categorías de claridad. Por otro lado, VS1 y VS2 presentan menos outliers que el resto. Además, todas las categorías de claridad presentan precios máximos y mínimos similares.

### Histogramas de variables numericas

In [ ]:
# 'Automatizar' la generación de gráficos con for loop

for col in numericas:
    fig = px.histogram(
        df,
        x=col,
        nbins=30,
        title=f'Histograma de {col}'
    )
    fig.show()

- ¿Qué variables numéricas muestran distribución sesgada?


### Mapa de calor con correlaciones

### Detección de outliers utilizando RIC (IQR)

In [ ]:
var = 'price'

Q1 = df[var].quantile(0.25)
Q3 = df[var].quantile(0.75)
IQR = Q3 - Q1

outliers = df[(df[var] < Q1 - 1.5 * IQR) | (df[var] > Q3 + 1.5 * IQR)]
print(f"Número de outliers en '{var}': {len(outliers)}")

### Análisis de Distribución

In [ ]:
from scipy.stats import skew, kurtosis

for col in numericas:
    print(f"{col}: Sesgo = {skew(df[col]):.2f}, Curtosis = {kurtosis(df[col]):.2f}")

### Transformación de variables categóricas

#### Label Encoding

In [ ]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['cut_encoded'] = le.fit_transform(df['cut'])

# Comparar valores originales vs codificados
df[['cut', 'cut_encoded']].drop_duplicates().sort_values('cut_encoded')

El órden lógico de esta columna es Fair < Good < Very Good < Ideal < Premium

In [ ]:
cut_order = {
    'Fair': 0,
    'Good': 1,
    'Very Good': 2,
    'Ideal': 3,
    'Premium': 4
}

df['cut_encoded'] = df['cut'].map(cut_order)

# Comparar valores originales vs codificados
df[['cut', 'cut_encoded']].drop_duplicates().sort_values('cut_encoded')

#### Ordinal Encoder

In [ ]:
# Ordinal Encoder
from sklearn.preprocessing import OrdinalEncoder

categories = [['Fair', 'Good', 'Very Good', 'Ideal', 'Premium']]

# Inicializamos el encoder con las categorías ordenadas
encoder = OrdinalEncoder(categories=categories)

# Ajustamos y transformamos
df['color_encoded'] = encoder.fit_transform(df[['color']])

# Comparar valores originales vs codificados
print(df[['color', 'color_encoded']].drop_duplicates().sort_values('color_encoded'))


#### One Hot Encoder

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Crear el codificador
ohe = OneHotEncoder(
    sparse_output=False, 
    drop='first'
)  # drop='first' para evitar multicolinealidad
ohe

In [ ]:
# Ajustar y transformar
color_encoded = ohe.fit_transform(df[['color']])
color_encoded

In [ ]:
# Crear DataFrame con nombres de columnas
color_cols = ohe.get_feature_names_out(['color'])

df_color_encoded = pd.DataFrame(
    color_encoded, 
    columns=color_cols
)

# Concatenar al DataFrame original
df = pd.concat(
    [df, df_color_encoded], 
    axis=1
)

# Mostrar mapeo de color
df[['color'] + list(color_cols)].drop_duplicates().sort_values('color')

In [ ]:
# Una manera MUCHO más fácil ... 
# En casos académicos, este método es mejor, sólo tendríamos que asignar este resultado a nuestro df existente
# Para producción o entornos más profesionales OneHotEncoder es mejor, dado a que puedes guardar y reutilizar el encoder
# basado en tus datos de entrenamiento

pd.get_dummies(df, columns=['color'], drop_first=True, dtype='int')

#### Target Encoder

In [ ]:
# Target Encoder
import category_encoders as ce

# Crear el codificador
te = ce.TargetEncoder(cols=['clarity'])

In [ ]:
# Ajustar y transformar
df['clarity_encoded'] = te.fit_transform(
    df['clarity'], 
    df['price']
)

# Comparar valores originales vs codificados
df[['clarity', 'clarity_encoded']].drop_duplicates().sort_values('clarity_encoded')

In [ ]:
df

### Escalamiento de Datos

### ¿Qué es el escalamiento de datos?

El escalamiento de datos es una técnica que ajusta los valores numéricos para que estén en rangos comparables. Esto es útil cuando usamos algoritmos que son sensibles a las magnitudes de los datos (como regresiones o clustering).

A continuación, se explican tres métodos comunes:

---

#### 🔹 MinMaxScaler

- Ajusta los valores para que estén entre un mínimo y un máximo (por defecto, entre 0 y 1).
- Fórmula:  
  $$
  X_{\text{escalado}} = \frac{X - X_{\text{min}}}{X_{\text{max}} - X_{\text{min}}}
  $$
- Útil cuando queremos conservar la forma original de la distribución.

---

#### 🔹 StandardScaler

- Centra los datos en 0 y los escala para que tengan una desviación estándar de 1.
- Fórmula:  
  $$
  X_{\text{escalado}} = \frac{X - \mu}{\sigma}
  $$
  donde $\mu$ es la media y $\sigma$ la desviación estándar.
- Ideal cuando los datos tienen una distribución aproximadamente normal.

---

En resumen: estos métodos ayudan a que los datos sean más comparables y útiles para análisis estadísticos o modelos de machine learning.

### ¿Se pueden combinar distintos escaladores?

Sí, es válido aplicar diferentes métodos de escalamiento a distintas columnas, especialmente cuando:

- Las variables tienen **naturaleza distinta** (por ejemplo, ingresos vs. proporciones).
- Algunas columnas tienen **valores extremos (outliers)** y otras no.
- Quieres conservar la **forma original** de ciertas distribuciones (MinMaxScaler) pero estandarizar otras (StandardScaler).

Ejemplo práctico:
- Usar `StandardScaler` en columnas como "ingresos" o "edad", que tienen distribución normal.
- Usar `MinMaxScaler` en columnas como "porcentaje de cumplimiento" o "calificaciones", que ya están en rangos definidos.

---

### Consideraciones importantes

- Si usas modelos **basados en distancia** (como KNN o clustering), asegúrate de que las escalas no generen sesgos. En ese caso, es mejor que todas las variables estén en rangos comparables.
- Para modelos como **árboles de decisión o random forest**, el escalamiento no es necesario, ya que no dependen de magnitudes.

---

En resumen: puedes escalar columnas de forma diferente, pero asegúrate de que tenga sentido para el tipo de análisis o modelo que estás usando.

# Pruebas de Hipótesis: Una y Dos Medias

Las **pruebas de hipótesis** nos ayudan a tomar decisiones sobre una población usando datos de una muestra. En particular, podemos comparar medias para saber si hay diferencias significativas o si los resultados podrían deberse al azar.

## ¿Qué es el estadístico t y el valor p?

- **Estadístico t:** Es una medida que compara la diferencia observada entre medias (o entre una media y un valor de referencia) con la variabilidad de los datos. Nos dice cuántas "desviaciones estándar" está la diferencia observada respecto a lo que esperaríamos por azar.
- **Valor p:** Es la probabilidad de obtener un resultado igual o más extremo que el observado, suponiendo que la hipótesis nula es cierta. Si el valor p es pequeño (por ejemplo, menor a 0.05), consideramos que la diferencia es significativa.

### Fórmulas

**Para una muestra:**

$$
t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}
$$

donde:

- $\bar{x}$ = media muestral  
- $\mu_0$ = media bajo la hipótesis nula  
- $s$ = desviación estándar muestral  
- $n$ = tamaño de la muestra  

**Para dos muestras independientes:**

$$
t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}
$$

donde los subíndices 1 y 2 corresponden a cada grupo.

---

In [ ]:
# Prueba de hipótesis para una media (t-test de una muestra)
from scipy.stats import ttest_1samp

media_hipotetica = 4000
stat, pvalue = ttest_1samp(df['price'], popmean=media_hipotetica)
print(f'Estadístico t: {stat:.2f}, p-valor: {pvalue:.4f}')

if pvalue < 0.05:
    print("Rechazamos H0: La media es significativamente diferente de 4000.")
else:
    print("No se rechaza H0: No hay evidencia suficiente para decir que la media es diferente de 4000.")

## 2. Prueba de Hipótesis para dos Medias

Se utiliza para comparar si las medias de dos grupos son iguales.

**Ejemplo:**  
¿El precio promedio de los diamantes con corte 'Ideal' es igual al de los de corte 'Premium'?

- **Hipótesis nula ($H_0$):** $\mu_1 = \mu_2$
- **Hipótesis alternativa ($H_1$):** $\mu_1 \neq \mu_2$

In [ ]:
# Prueba de hipótesis para dos medias independientes (t-test de dos muestras)
from scipy.stats import ttest_ind

grupo1 = df[df['cut'] == 'Ideal']['price']
grupo2 = df[df['cut'] == 'Premium']['price']

stat, pvalue = ttest_ind(grupo1, grupo2, equal_var=False)
print(f'Estadístico t: {stat:.2f}, p-valor: {pvalue:.4f}')

if pvalue < 0.05:
    print("Rechazamos H0: Las medias de precio son significativamente diferentes entre 'Ideal' y 'Premium'.")
else:
    print("No se rechaza H0: No hay evidencia suficiente para decir que las medias son diferentes.")